# Real-world data coding for neuroscientists (ReCoN)

### MSc in Translational Neuroscience,

### Department of Brain Sciences, Faculty of Medicine,

### Imperial College London

### Autumn 2026

# End-of-day task: linking wearable sensor data to health records

<div style="background-color:transparent; border-left:5px solid #5DADE2; padding:10px; margin-bottom:15px;">

Today you worked with two kinds of real-world data: **tabular health records** (pandas tutorial) and **time-series signals** (time-series tutorial). In research they are often used together: a sensor recording could be turned into a few numbers (**features**), and those features are linked to each patient's clinical record.

**The scenario.** The six patients from the EHR tutorial wore a wrist accelerometer during a **20-second rest recording** at their **most recent clinic visit**. Tremor at rest is one of the hallmark motor signs of Parkinson's disease, and it typically oscillates at about **4–6 Hz**. Your job:

| Part | What you do | Time |
|:--|:--|:--|
| **A** | Keep each patient's most recent visit from the health records | ≈ 7 min |
| **B** | Clean the recordings and turn each one into two features: **tremor amplitude** and **dominant frequency** | ≈ 15 min |
| **C** | Combine both tables, make one plot and interpret it | ≈ 8 min |

</div>

<div style="background-color:transparent; border-left:5px solid #48C9B0; padding:15px; border-radius:8px;">

<h3 style="color:#2C3E50;">The data</h3>

**1. `data/ehr_parkinsons.csv`**: the same synthetic health records as in the pandas tutorial (one row per clinic visit: `patient_id`, `sex`, `date_of_birth`, `visit_date`, `clinic_id`, `updrs_iii`, `ledd_mg`, `medication`, `bmi`).

**2. `data/wearable_rest_recordings.csv`**: one row per **sample** of the wrist accelerometer.

| **Column** | **Description** |
|:--|:--|
| **`patient_id`** | Patient identifier (the same IDs as in the health records) |
| **`t_s`** | Time since the start of the recording, in seconds |
| **`Acc_X`**, **`Acc_Y`**, **`Acc_Z`** | Acceleration along the x, y and z axes, in **g** units |
| **`fs_hz`** | Sampling frequency in Hz (100 samples per second) |

Both files are synthetic (made up), but they contain the kind of problems you have learned to deal with today.

</div>

In [ ]:
# NOTE: don't worry about this cell. Run it once: it creates the two data files for this task.
import os
import numpy as np
import pandas as pd

os.makedirs("end_of_day_data", exist_ok=True)

# 1. Health records (same as in the pandas tutorial)
pd.DataFrame({
    "patient_id": ["P01", "P02", "P03", "P04", "P05", "P01", "P02", "P03", "P05", "P06", "P01", "P02", "P05", "P04"],
    "sex": ["Male", "Female", "Female", "Male", "Male", "Male", "Female", "Female", "Male", "Female", "Male", "Female", "Male", "Male"],
    "date_of_birth": ["1950-05-14", "1965-08-22", "1970-11-30", "1980-01-15", "1958-03-02", "1950-05-14", "1965-08-22",
                      "1970-11-30", "1958-03-02", "1955-12-01", "1950-05-14", "1965-08-22", "1958-03-02", "1980-01-15"],
    "visit_date": ["2023-01-10", "2023-01-15", "2023-02-01", "2023-03-14", "2023-04-21", "2023-07-12", "2023-09-03",
                   "2023-10-06", "2023-11-30", "2024-01-09", "2024-02-08", "2024-03-20", "2024-05-17", "2024-06-04"],
    "clinic_id": [101, 102, 101, 103, 102, 101, 102, 101, 102, 103, 101, 102, 102, 103],
    "updrs_iii": [22, 30, 40, 12, 35, 25, None, 42, 38, 27, 28, 31, 41, None],
    "ledd_mg": [300, 150, 800, 0, 450, 400, 300, 850, 600, 400, 500, 300, 750, 0],
    "medication": ["Co-careldopa", "Rotigotine", "Co-beneldopa", None, "Pramipexole", "co-careldopa ", "rotigotine",
                   "co-beneldopa", "Co-careldopa + Entacapone", "Ropinirole", "CO-CARELDOPA", " Rotigotine",
                   "co-careldopa + entacapone ", None],
    "bmi": ["22.5", "25.0", "30.0", "28.0", "26.4", "22.7", "not recorded", "31.2", "26.1", "21.9", "23.1", "24.0",
            "not recorded", "27.6"],
}).to_csv("data/ehr_parkinsons.csv", index=False)

# 2. Wrist accelerometer: 20 s at rest, 100 Hz, one recording per patient
rng = np.random.default_rng(seed=7)
fs = 100
t = np.arange(20 * fs) / fs
tremor_settings = {"P01": (5.0, 0.035), "P02": (4.4, 0.025), "P03": (5.4, 0.090),
                   "P04": (0.0, 0.000), "P05": (4.8, 0.070), "P06": (5.8, 0.030)}   # (frequency in Hz, amplitude in g)
recordings = []
for pid, (freq, amp) in tremor_settings.items():
    phase = rng.uniform(0, 2 * np.pi)
    tremor = amp * (1 + 0.3 * np.sin(2 * np.pi * 0.1 * t + phase)) * np.sin(2 * np.pi * freq * t + phase)
    drift = 0.02 * np.sin(2 * np.pi * 0.15 * t + phase)          # slow changes in wrist posture
    rec = pd.DataFrame({
        "patient_id": pid,
        "t_s": np.round(t, 2),
        "Acc_X": 0.10 + 0.8 * tremor + drift + rng.normal(0, 0.005, t.size),
        "Acc_Y": 0.15 + 0.4 * tremor + rng.normal(0, 0.005, t.size),
        "Acc_Z": 0.98 + 0.6 * tremor + rng.normal(0, 0.005, t.size),
        "fs_hz": fs,
    })
    if pid == "P03":   # a 0.6-second dropout
        rec.loc[800:859, ["Acc_X", "Acc_Y", "Acc_Z"]] = np.nan
    if pid == "P05":   # 20 scattered missing samples
        rec.loc[rng.choice(np.arange(10, 1990), 20, replace=False), ["Acc_X", "Acc_Y", "Acc_Z"]] = np.nan
    recordings.append(rec)

pd.concat(recordings, ignore_index=True).round({"Acc_X": 4, "Acc_Y": 4, "Acc_Z": 4}).to_csv(
    "data/wearable_rest_recordings.csv", index=False)

print("Files created in the folder 'end_of_day_data'!")

Files created in the folder 'end_of_day_data'!


In [ ]:
# Run this cell: the libraries you need for the task
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Part A: the health records (≈ 7 min)

The recordings were made at each patient's **most recent** visit, so we only need that visit from the health records.

In [ ]:
#  TASK PART A
# 1. Load "data/ehr_parkinsons.csv" into a variable called `ehr`, and convert the "visit_date" column to dates.     (pandas 2.1.2, 2.3.2)
# 
# 2. Keep only each patient's MOST RECENT visit:
#    a) add a column "last_visit" with each patient's latest visit date, repeated on all their visits               (pandas 2.5.4)
#    b) keep only the rows where "visit_date" is equal to "last_visit". Call the result `ehr_last`.                 (pandas 2.2.3)
# 
# 3. Keep only the columns "patient_id", "visit_date", "updrs_iii", "ledd_mg" and "medication", and display `ehr_last`. You should have 6 rows: one per patient.      (pandas 2.2.1)

# Write your code here:

## Part B: from signals to features (≈ 15 min)

You will describe each 20-second recording with two **features**, calculated from the signal vector magnitude:

- **Tremor amplitude**: the standard deviation of the vector magnitude. The stronger the tremor, the more the signal oscillates around its average.
- **Tremor IQR**: The interquartile range of the vector magnitude.

In [ ]:
#  TASK PART B
# 1. Load "data/wearable_rest_recordings.csv" into a variable called `rec`.
#    How many rows and how many different patients are there? How many seconds does each recording last?
#    (fs = 100 Hz, so seconds = number of samples / 100)                                       (pandas 2.1.3, 2.3.4, 2.4.1)


# 2. Count the missing values in each column, and find which patient(s) they belong to.
#    (filter the rows where "Acc_X" is missing, then use .value_counts() on "patient_id")      (pandas 2.3.3, 2.3.4)


# 3. Fill in the gaps in the three acceleration columns ("Acc_X", "Acc_Y", "Acc_Z"), using EITHER
#      - forward fill (each gap takes the last value before it), OR
#      - linear interpolation (each gap is filled with a straight line between the values on either side).
#    Then check that no missing values are left.                                                                              (we covered this in time-series notebook: ECG → Remove NaNs)
#    HINT: in pandas each option is ONE command applied to the columns, e.g. rec[["Acc_X", "Acc_Y", "Acc_Z"]].something()
#          Google "pandas DataFrame ffill" or "pandas DataFrame interpolate", or read:
#          https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html
#          https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html


# 4. Add a column "vector_mag" with the signal vector magnitude.                                (we covered this in time-series notebook: time series: accelerometer → final task)


# 5. Create a table called `features` with ONE ROW PER PATIENT and two columns:                       (pandas 2.5.3)
#      "tremor_amplitude":  the standard deviation ("std") of "vector_mag"
#      "tremor IQR": the interquartile range of the vector magnitude, the distance between the 75th and 25th percentiles. Like the std it measures spread, but it ignores extreme values (time-series tutorial B.7a).
#    Use .reset_index() so that "patient_id" is a normal column and not an index column.
#    HINT: .agg() also accepts YOUR OWN function instead of a name like "max", e.g. ("vector_mag", dominant_frequency)
#          (no brackets after the function name, like with .apply()).
#          read the section on aggregation in: https://pandas.pydata.org/docs/user_guide/groupby.html

# Write your code here:

## Part C: combine and interpret (≈ 8 min)

In [ ]:
#  TASK PART C
# 1. Merge `features` with `ehr_last` using "patient_id" (a left merge), call the result `combined`, and display it.        (pandas 2.4.2)
#
#
# 2. Make a scatter plot with "tremor_amplitude" on the x axis and "updrs_iii" on the y axis.
#    Label both axes and add a title.                                       (time series: plotting with plt)
#    HINT: plt.scatter(x_values, y_values) works like plt.plot(), but draws only points, without lines.
#          Google "matplotlib pyplot scatter", or read:
#          https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html
#
#
# 3. Answer the questions in the cell below.
#
#
# 4. Add a True/False column "pd_tremor_range" to `combined` that is True when "dominant_freq_hz" is between 4 and 6 Hz.
#    Use .apply() with a lambda.                                                                   (pandas 2.5.2)

# Write your code here:

##### What do you observe?

1. Which patient has the strongest tremor? Which medication and dose (`ledd_mg`) were they on at that visit?
2. Do higher UPDRS III scores go together with stronger tremor? With only five points on the plot, how confident can you be? Remember that UPDRS III scores many motor signs, not only tremor (e.g. rigidity, slowness of movement, gait).
3. One patient does not appear on your plot. Who, and why?

<br>